In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/workspace/tiny-llm-from-scratch")

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/workspace/tiny-llm-from-scratch


In [5]:
import torch

from tokenizer.bpe_tokenizer import BPETokenizer

from training.dataset import TinyDataset
from training.sequence import SequenceBuilder

from training.data_loader import (
    TinyTorchDataset,
    TinyDataLoader,
)

In [6]:
dataset = TinyDataset.load(
    PROJECT_ROOT /
    "datasets/processed/split/train.json"
)

print("Dataset Samples:", len(dataset))

Dataset Samples: 4783


In [8]:
tokenizer = BPETokenizer.load(
    PROJECT_ROOT /
    "tokenizer/bpe/tokenizer.json"
)

print("Tokenizer Loaded")

Tokenizer Loaded


In [9]:
builder = SequenceBuilder(
    tokenizer=tokenizer
)

print("SequenceBuilder Ready")

SequenceBuilder Ready


In [10]:
encoded_samples = []

for sample in dataset.head(20):

    encoded_samples.append(
        builder.encode_sample(sample)
    )

print("Encoded Samples:", len(encoded_samples))

Encoded Samples: 20


In [11]:
training_pairs = builder.build_dataset(
    encoded_samples,
    block_size=64,
)

print("Training Pairs:", len(training_pairs))

Training Pairs: 16413


In [12]:
torch_dataset = TinyTorchDataset(
    training_pairs
)

print(torch_dataset)

In [13]:
print(len(torch_dataset))

16413


In [14]:
x, y = torch_dataset[0]

print(type(x))
print(type(y))

<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [15]:
print(x.shape)

print(y.shape)

torch.Size([64])
torch.Size([64])


In [16]:
print(x.dtype)

print(y.dtype)

torch.int64
torch.int64


In [19]:
from training.data_loader import DataLoaderConfig

config = DataLoaderConfig(
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

loader = TinyDataLoader(
    dataset=torch_dataset,
    config=config,
)

train_loader = loader.create_train_loader()

loader.summary()

Tiny DataLoader
Dataset Samples : 16413
Batch Size      : 8
Shuffle         : True
Num Workers     : 0
Pin Memory      : False
Drop Last       : False
Batches         : 2052


In [21]:
batch = next(iter(train_loader))

batch_x, batch_y = batch

print(batch_x.shape)

print(batch_y.shape)

torch.Size([8, 64])
torch.Size([8, 64])


In [22]:
print(batch_x[0])

print()

print(batch_y[0])

tensor([11,  8,  6,  9,  5, 13, 11, 13, 11,  5,  6, 14,  5,  7, 15, 13,  5,  7,
        20,  8, 13,  8,  8,  6, 12, 12, 11,  6, 13,  8, 14,  5, 10,  8,  7,  8,
         6,  9,  6,  5, 10,  5, 10, 16, 13, 13,  7, 15,  6,  5, 14, 12,  5, 10,
         8,  8, 13,  8, 12, 14,  8, 14, 10, 16])

tensor([ 8,  6,  9,  5, 13, 11, 13, 11,  5,  6, 14,  5,  7, 15, 13,  5,  7, 20,
         8, 13,  8,  8,  6, 12, 12, 11,  6, 13,  8, 14,  5, 10,  8,  7,  8,  6,
         9,  6,  5, 10,  5, 10, 16, 13, 13,  7, 15,  6,  5, 14, 12,  5, 10,  8,
         8, 13,  8, 12, 14,  8, 14, 10, 16, 12])


In [23]:
assert batch_x.shape[0] == 8
assert batch_y.shape[0] == 8

assert batch_x.shape[1] == 64
assert batch_y.shape[1] == 64

assert batch_x.dtype == torch.long
assert batch_y.dtype == torch.long

print("✓ Assertions Passed")

✓ Assertions Passed


In [24]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():

    device = torch.device("cuda")

else:

    device = torch.device("cpu")

print(device)

CUDA Available: False
cpu


In [25]:
batch_x = batch_x.to(device)

batch_y = batch_y.to(device)

print(batch_x.device)

print(batch_y.device)

cpu
cpu


In [26]:
print("=" * 60)

print("TinyTorchDataset")

print("=" * 60)

print("Dataset Length :", len(torch_dataset))

print("Batch Size     :", batch_x.shape[0])

print("Sequence Length:", batch_x.shape[1])

print("Device         :", device)

TinyTorchDataset
Dataset Length : 16413
Batch Size     : 8
Sequence Length: 64
Device         : cpu


In [27]:
print("=" * 60)

print("TinyLLM DataLoader Validation")

print("=" * 60)

print("Dataset Samples :", len(dataset))

print("Training Pairs  :", len(training_pairs))

print("Torch Dataset   :", len(torch_dataset))

print("Batch Size      :", batch_x.shape[0])

print("Sequence Length :", batch_x.shape[1])

print("Tensor Type     :", batch_x.dtype)

print()

print("★★★★★ Phase 11.06 PASSED")

print("=" * 60)

TinyLLM DataLoader Validation
Dataset Samples : 4783
Training Pairs  : 16413
Torch Dataset   : 16413
Batch Size      : 8
Sequence Length : 64
Tensor Type     : torch.int64

★★★★★ Phase 11.06 PASSED
